# Pjesa_2 — Sisteme dinamike klasike dhe ODE  
## Notebook: `01_sisteme_lagranzhiane_dhe_ode_te_nderlikuara.ipynb`

Ky notebook është menduar për bllokun **Pjesa_2: Sisteme dinamike klasike dhe ODE**.  
Ai ndërton një urë të qartë midis:

- modelimit fizik;
- formalizmit të **Lagranzhianit**;
- derivimit simbolik me **SymPy**;
- kalimit në sistem ODE të rendit të parë;
- zgjidhjes numerike me **SciPy**;
- interpretimit grafik dhe animacionit.

Në vijim studiohen dy sisteme klasike:

1. **Mekanizmi bielë-manivelë**  
2. **Penduli i dyfishtë**

Në fund shtohet edhe një shembull i shkurtër që tregon se të njëjtat metoda numerike mund të përdoren edhe **jashtë fushës së lëkundjeve**, si dhe disa **ushtrime të propozuara** për studentët.

> **Shënim terminologjik:** në gjithë notebook-un përdoret forma **Lagranzhian**.

In [ ]:
# Bibliotekat bazë

import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from scipy.integrate import solve_ivp
from IPython.display import HTML, display

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True

## 1. Nga Lagranzhiani te ekuacionet e lëvizjes

Në mekanikën analitike, për një grup koordinatash të përgjithshme \(q_i\), Lagranzhiani përkufizohet si:

\[
L(q_i,\dot q_i,t)=T-V,
\]

ku:

- \(T\) është energjia kinetike;
- \(V\) është energjia potenciale.

Ekuacionet e Lagranzhit janë:

\[
\frac{d}{dt}\left(\frac{\partial L}{\partial \dot q_i}\right)-\frac{\partial L}{\partial q_i}=0.
\]

Në këtë notebook ndjekim gjithmonë të njëjtin zinxhir pune:

1. zgjedhim koordinatat e përgjithshme;
2. shkruajmë \(T\) dhe \(V\);
3. ndërtojmë **Lagranzhianin**;
4. nxjerrim ekuacionet me **SymPy**;
5. i kthejmë në ODE të rendit të parë;
6. i zgjidhim numerikisht;
7. i vizualizojmë dhe i animuojmë.

## 2. Importet simbolike

Në mënyrë që derivimet të mbeten sa më të qarta, më poshtë vendosen simbolet kryesore që do të përdoren në pjesën simbolike.

In [ ]:
t = sp.symbols('t', real=True)
g = sp.symbols('g', positive=True, real=True)

# 3. Sistemi I — Mekanizmi bielë-manivelë

## 3.1 Përshkrimi fizik

Mekanizmi **bielë-manivelë** është një sistem klasik i përdorur në motorë, pompa dhe shumë pajisje mekanike.  
Ai shndërron lëvizjen rrotulluese në lëvizje të drejtvizore reciproke.

Në modelin më të thjeshtë:

- manivela ka gjatësi \(r\);
- biela ka gjatësi \(l\);
- pistoni lëviz vetëm në drejtimin horizontal;
- koordinata e përgjithshme zgjidhet si këndi i manivelës \(\theta(t)\).

Pozicioni horizontal i pistonit është:

\[
x(	heta)=r\cos	heta+\sqrt{l^2-r^2\sin^2	heta}.
\]

Kjo lidhje shpreh kufizimin gjeometrik të mekanizmit.

## 3.2 Ndërtimi i Lagranzhianit

Për një model të thjeshtuar marrim:

- një moment inercie efektiv \(I\) për pjesën rrotulluese;
- një masë \(m\) për pistonin;
- një sustë rrotulluese me konstantë \(k\), që e bën problemin dinamik autonom.

Atëherë:

\[
T = rac12 I \dot	heta^2 + rac12 m \dot x^2,
\qquad
V = rac12 k 	heta^2.
\]

Prandaj:

\[
L = T - V.
\]

Ky model është mjaftueshëm i pasur për të treguar sesi një **kufizim gjeometrik jo-linear** prodhon ekuacione lëvizjeje jo-lineare.

In [ ]:
# Derivimi simbolik për mekanizmin bielë-manivelë

theta = sp.Function('theta')(t)
r, l, m, I, k = sp.symbols('r l m I k', positive=True, real=True)

x = r*sp.cos(theta) + sp.sqrt(l**2 - r**2 * sp.sin(theta)**2)
xdot = sp.diff(x, t)

T = sp.Rational(1,2)*I*sp.diff(theta, t)**2 + sp.Rational(1,2)*m*xdot**2
V = sp.Rational(1,2)*k*theta**2
L = sp.simplify(T - V)

EL_biele_manivele = sp.simplify(
    sp.diff(sp.diff(L, sp.diff(theta, t)), t) - sp.diff(L, theta)
)

EL_biele_manivele

## 3.3 Forma strukturore e ekuacionit

Pavarësisht se shprehja del e gjatë, ajo ka në thelb formën:

\[
M(	heta)\,\ddot	heta + C(	heta)\,\dot	heta^2 + k	heta = 0.
\]

Kjo është shumë e rëndësishme pedagogjikisht:  
edhe kur kemi vetëm **një koordinatë të përgjithshme**, kufizimet gjeometrike e bëjnë masën efektive dhe termat inercialë **funksione jo-lineare të pozicionit**.

Në qelizën vijuese, nxjerrim automatikisht \(M(	heta)\) dhe \(C(	heta)\).

In [ ]:
theta_s, omega_s = sp.symbols('theta_s omega_s', real=True)
thetadd_s = sp.symbols('thetadd_s', real=True)

expr = EL_biele_manivele.subs({
    theta: theta_s,
    sp.diff(theta, t): omega_s,
    sp.diff(theta, (t, 2)): thetadd_s
})

M_expr = sp.simplify(sp.diff(expr, thetadd_s))
rest_expr = sp.simplify(expr - M_expr*thetadd_s)
C_expr = sp.simplify(sp.expand(rest_expr - k*theta_s) / (omega_s**2))

display(sp.Eq(sp.Symbol('M(theta)'), M_expr))
display(sp.Eq(sp.Symbol('C(theta)'), C_expr))

## 3.4 Kalimi në ODE numerike

Do të përdorim parametrat numerikë:

- \(r = 0.10\,\mathrm{m}\)
- \(l = 0.30\,\mathrm{m}\)
- \(m = 1.0\,\mathrm{kg}\)
- \(I = 0.02\,\mathrm{kg\,m^2}\)
- \(k = 0.50\,\mathrm{N\,m/rad}\)

Për të bërë simulimin më realist, shtojmë edhe një amortizim të vogël viskoz \(c\dot\theta\), i cili nuk futet në Lagranzhian, por në ekuacionin e lëvizjes si forcë e përgjithshme jo-konservative:

\[
M(	heta)\,\ddot	heta + C(	heta)\,\dot	heta^2 + c\dot	heta + k	heta = 0.
\]

In [ ]:
# Lambdify për pjesën numerike

M_fun = sp.lambdify((theta_s, r, l, m, I), M_expr, 'numpy')
C_fun = sp.lambdify((theta_s, r, l, m, I), C_expr, 'numpy')
x_fun = sp.lambdify((theta_s, r, l), x.subs(theta, theta_s), 'numpy')

params_bm = {
    "r": 0.10,
    "l": 0.30,
    "m": 1.0,
    "I": 0.02,
    "k": 0.50,
    "c": 0.05,
}

def ode_biele_manivele(t, y, p):
    th, om = y
    Mv = M_fun(th, p["r"], p["l"], p["m"], p["I"])
    Cv = C_fun(th, p["r"], p["l"], p["m"], p["I"])
    thdd = -(Cv*om**2 + p["c"]*om + p["k"]*th) / Mv
    return [om, thdd]

In [ ]:
# Zgjidhja numerike

t_span_bm = (0.0, 25.0)
t_eval_bm = np.linspace(*t_span_bm, 2500)
y0_bm = [0.8, 0.0]   # theta(0), omega(0)

sol_bm = solve_ivp(
    ode_biele_manivele,
    t_span_bm,
    y0_bm,
    t_eval=t_eval_bm,
    args=(params_bm,),
    rtol=1e-8,
    atol=1e-10,
)

theta_bm = sol_bm.y[0]
omega_bm = sol_bm.y[1]
x_bm = x_fun(theta_bm, params_bm["r"], params_bm["l"])

## 3.5 Rezultatet grafike

Shikojmë:

1. evolucionin e këndit \(\theta(t)\);
2. shpejtësinë këndore \(\dot\theta(t)\);
3. zhvendosjen e pistonit \(x(t)\).

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(10, 10), sharex=True)

ax[0].plot(sol_bm.t, theta_bm)
ax[0].set_ylabel(r"$\theta(t)$ [rad]")
ax[0].set_title("Mekanizmi bielë-manivelë: variablat kryesore")

ax[1].plot(sol_bm.t, omega_bm)
ax[1].set_ylabel(r"$\dot\theta(t)$ [rad/s]")

ax[2].plot(sol_bm.t, x_bm)
ax[2].set_ylabel(r"$x(t)$ [m]")
ax[2].set_xlabel("Koha [s]")

plt.tight_layout()
plt.show()

## 3.6 Animacion i mekanizmit bielë-manivelë

Për animacionin përdorim **Matplotlib Animation**, e cila është një zgjidhje standarde dhe shumë e dobishme në notebook.  
Kjo qelizë vizualizon:

- boshtin e rrotullimit të manivelës;
- majën e manivelës;
- bielën;
- pistonin në shinë.

Nëse notebook-u ekzekutohet në Jupyter, animacioni shfaqet brenda faqes.

In [ ]:
from matplotlib.animation import FuncAnimation

r_num = params_bm["r"]
l_num = params_bm["l"]

x_crank = r_num * np.cos(theta_bm)
y_crank = r_num * np.sin(theta_bm)
x_slider = x_bm
y_slider = np.zeros_like(x_slider)

fig, ax = plt.subplots(figsize=(9, 4))
ax.set_aspect("equal")
ax.set_xlim(-0.15, 0.45)
ax.set_ylim(-0.18, 0.18)
ax.set_title("Animacion: mekanizmi bielë-manivelë")
ax.plot([-0.15, 0.45], [0, 0], 'k--', alpha=0.4)

line_crank, = ax.plot([], [], 'o-', lw=3)
line_rod, = ax.plot([], [], 'o-', lw=3)
slider_point, = ax.plot([], [], 's', ms=10)
trace_slider, = ax.plot([], [], lw=1, alpha=0.5)

def init_bm():
    line_crank.set_data([], [])
    line_rod.set_data([], [])
    slider_point.set_data([], [])
    trace_slider.set_data([], [])
    return line_crank, line_rod, slider_point, trace_slider

def update_bm(i):
    xc, yc = x_crank[i], y_crank[i]
    xs, ys = x_slider[i], y_slider[i]

    line_crank.set_data([0, xc], [0, yc])
    line_rod.set_data([xc, xs], [yc, ys])
    slider_point.set_data([xs], [ys])
    trace_slider.set_data(x_slider[:i+1], y_slider[:i+1])

    return line_crank, line_rod, slider_point, trace_slider

step_bm = max(1, len(sol_bm.t)//250)
ani_bm = FuncAnimation(
    fig, update_bm, frames=range(0, len(sol_bm.t), step_bm),
    init_func=init_bm, interval=40, blit=True
)

plt.close(fig)
HTML(ani_bm.to_jshtml())

# 4. Sistemi II — Penduli i dyfishtë

## 4.1 Përshkrimi fizik

Penduli i dyfishtë përbëhet nga dy masa të lidhura me dy shufra të ngurta.  
Si koordinata të përgjithshme zgjedhim:

- \(\theta_1(t)\): këndi i shufrës së parë me vertikalen;
- \(\theta_2(t)\): këndi i shufrës së dytë me vertikalen.

Ky sistem është një shembull klasik i:

- jo-linearitetit;
- bashkëveprimit midis koordinatave;
- ndjeshmërisë ndaj kushteve fillestare.

Në këtë notebook ne **nuk theksojmë aspektin kaotik**, sepse ai do të trajtohet në blloqet vijuese.  
Këtu interesi ynë është: **si ndërtohen ekuacionet dhe si zgjidhen numerikisht**.

## 4.2 Lagranzhiani i pendulit të dyfishtë

Le të jenë:

- \(m_1, m_2\) masat;
- \(L_1, L_2\) gjatësitë e shufrave.

Pozicionet karteziane janë:

\[
x_1=L_1\sin\theta_1,\qquad y_1=-L_1\cos\theta_1,
\]

\[
x_2=x_1+L_2\sin\theta_2,\qquad y_2=y_1-L_2\cos\theta_2.
\]

Prej tyre ndërtohen \(T\) dhe \(V\), dhe më pas:

\[
L=T-V.
\]

Për të shmangur gabimet algjebrike dhe për ta bërë procesin më të përgjithshëm, përdorim **SymPy Mechanics**.

In [ ]:
import sympy.physics.mechanics as me

# Simbolet dhe funksionet
m1, m2, L1, L2 = sp.symbols('m1 m2 L1 L2', positive=True, real=True)
th1, th2 = me.dynamicsymbols('th1 th2')

# Kornizat
N = me.ReferenceFrame('N')
A = N.orientnew('A', 'Axis', [th1, N.z])
B = N.orientnew('B', 'Axis', [th2, N.z])

# Pikat
O = me.Point('O')
O.set_vel(N, 0)

P1 = O.locatenew('P1', L1*A.x)
P2 = P1.locatenew('P2', L2*B.x)

P1.v2pt_theory(O, N, A)
P2.v2pt_theory(P1, N, B)

# Grimcat
Pa1 = me.Particle('Pa1', P1, m1)
Pa2 = me.Particle('Pa2', P2, m2)

# Energjitë
T_dp = sp.simplify(Pa1.kinetic_energy(N) + Pa2.kinetic_energy(N))
V_dp = m1*g*P1.pos_from(O).dot(N.y) + m2*g*P2.pos_from(O).dot(N.y)

L_dp = me.Lagrangian(N, Pa1, Pa2) - V_dp  # L = T - V
LM = me.LagrangesMethod(L_dp, [th1, th2])
eqs_dp = sp.simplify(LM.form_lagranges_equations())

eqs_dp

## 4.3 Matrica e masës dhe termat e forcimit

Një avantazh i madh i **SymPy Mechanics** është se i shkruan ekuacionet si sistem linear në \(\ddot\theta_1\) dhe \(\ddot\theta_2\):

\[
\mathbf{M}(q)\,\ddot{\mathbf q} = \mathbf{f}(q,\dot q).
\]

Ky format është ideal për integrimin numerik.

In [ ]:
M_dp = sp.simplify(LM.mass_matrix)
F_dp = sp.simplify(LM.forcing)

display(M_dp)
display(F_dp)

## 4.4 Kalimi në formë numerike

Parametrat numerikë:

- \(m_1 = m_2 = 1.0\,\mathrm{kg}\)
- \(L_1 = L_2 = 1.0\,\mathrm{m}\)
- \(g = 9.81\,\mathrm{m/s^2}\)

Kushtet fillestare zgjidhen të moderuara, në mënyrë që lëvizja të jetë e pasur, por pa hyrë ende në analizë të hollësishme të kaosit.

In [ ]:
# Lambdify për pendulin e dyfishtë

th1_s, th2_s, w1_s, w2_s = sp.symbols('th1_s th2_s w1_s w2_s', real=True)

M_dp_num_expr = M_dp.subs({
    th1: th1_s,
    th2: th2_s,
    sp.diff(th1, t): w1_s,
    sp.diff(th2, t): w2_s
})

F_dp_num_expr = F_dp.subs({
    th1: th1_s,
    th2: th2_s,
    sp.diff(th1, t): w1_s,
    sp.diff(th2, t): w2_s
})

M_dp_fun = sp.lambdify((th1_s, th2_s, w1_s, w2_s, m1, m2, L1, L2, g), M_dp_num_expr, 'numpy')
F_dp_fun = sp.lambdify((th1_s, th2_s, w1_s, w2_s, m1, m2, L1, L2, g), F_dp_num_expr, 'numpy')

params_dp = {
    "m1": 1.0,
    "m2": 1.0,
    "L1": 1.0,
    "L2": 1.0,
    "g": 9.81
}

def ode_double_pendulum(t, y, p):
    th1v, w1v, th2v, w2v = y
    Mv = np.array(M_dp_fun(th1v, th2v, w1v, w2v, p["m1"], p["m2"], p["L1"], p["L2"], p["g"]), dtype=float)
    Fv = np.array(F_dp_fun(th1v, th2v, w1v, w2v, p["m1"], p["m2"], p["L1"], p["L2"], p["g"]), dtype=float).reshape(2)
    acc = np.linalg.solve(Mv, Fv)
    return [w1v, acc[0], w2v, acc[1]]

In [ ]:
# Zgjidhja numerike për pendulin e dyfishtë

t_span_dp = (0.0, 20.0)
t_eval_dp = np.linspace(*t_span_dp, 4000)

y0_dp = [0.8, 0.0, 1.1, 0.0]  # theta1, omega1, theta2, omega2

sol_dp = solve_ivp(
    ode_double_pendulum,
    t_span_dp,
    y0_dp,
    t_eval=t_eval_dp,
    args=(params_dp,),
    rtol=1e-9,
    atol=1e-9
)

theta1_dp = sol_dp.y[0]
omega1_dp = sol_dp.y[1]
theta2_dp = sol_dp.y[2]
omega2_dp = sol_dp.y[3]

## 4.5 Grafiqet kryesore

Në vijim paraqiten:

- këndet \(\theta_1(t)\) dhe \(\theta_2(t)\);
- shpejtësitë këndore \(\dot\theta_1(t)\) dhe \(\dot\theta_2(t)\).

Qëllimi këtu është të shihet ndërveprimi midis dy koordinatave të përgjithshme.

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

ax[0].plot(sol_dp.t, theta1_dp, label=r"$\theta_1(t)$")
ax[0].plot(sol_dp.t, theta2_dp, label=r"$\theta_2(t)$")
ax[0].set_ylabel("Këndi [rad]")
ax[0].set_title("Penduli i dyfishtë: këndet")
ax[0].legend()

ax[1].plot(sol_dp.t, omega1_dp, label=r"$\dot\theta_1(t)$")
ax[1].plot(sol_dp.t, omega2_dp, label=r"$\dot\theta_2(t)$")
ax[1].set_ylabel("Shpejtësia këndore [rad/s]")
ax[1].set_xlabel("Koha [s]")
ax[1].legend()

plt.tight_layout()
plt.show()

## 4.6 Trajektorja e masës së dytë

Një grafik shumë ilustrues është trajektorja e masës së dytë në plan.  
Ky grafik i ndihmon studentët të kuptojnë se sjellja hapësinore e sistemit është dukshëm më e pasur sesa ajo e një penduli të vetëm.

In [ ]:
L1_num = params_dp["L1"]
L2_num = params_dp["L2"]

x1_dp = L1_num * np.sin(theta1_dp)
y1_dp = -L1_num * np.cos(theta1_dp)

x2_dp = x1_dp + L2_num * np.sin(theta2_dp)
y2_dp = y1_dp - L2_num * np.cos(theta2_dp)

plt.figure(figsize=(6, 6))
plt.plot(x2_dp, y2_dp, lw=1.0)
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Trajektorja e masës së dytë")
plt.axis("equal")
plt.show()

## 4.7 Animacion i avancuar me Plotly

Për një vizualizim më modern dhe interaktiv përdorim **Plotly**, e cila mbështet:

- slider kohor;
- butona play/pause;
- kontroll interaktiv në notebook.

Nëse Plotly nuk është instaluar në mjedisin tuaj, qeliza kap gabimin dhe jep një njoftim të qartë.

In [ ]:
try:
    import plotly.graph_objects as go

    step_dp = max(1, len(sol_dp.t)//250)

    frames = []
    for i in range(0, len(sol_dp.t), step_dp):
        frames.append(
            go.Frame(
                data=[
                    go.Scatter(
                        x=[0, x1_dp[i], x2_dp[i]],
                        y=[0, y1_dp[i], y2_dp[i]],
                        mode="lines+markers"
                    ),
                    go.Scatter(
                        x=x2_dp[:i+1],
                        y=y2_dp[:i+1],
                        mode="lines"
                    )
                ],
                name=str(i)
            )
        )

    fig = go.Figure(
        data=[
            go.Scatter(x=[0, x1_dp[0], x2_dp[0]], y=[0, y1_dp[0], y2_dp[0]], mode="lines+markers"),
            go.Scatter(x=[x2_dp[0]], y=[y2_dp[0]], mode="lines")
        ],
        frames=frames
    )

    fig.update_layout(
        title="Penduli i dyfishtë — animacion interaktiv",
        xaxis=dict(range=[-2.2, 2.2], autorange=False),
        yaxis=dict(range=[-2.2, 1.2], autorange=False, scaleanchor="x", scaleratio=1),
        updatemenus=[{
            "type": "buttons",
            "buttons": [
                {"label": "Play", "method": "animate",
                 "args": [None, {"frame": {"duration": 35, "redraw": True}, "fromcurrent": True}]},
                {"label": "Pause", "method": "animate",
                 "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]}
            ]
        }]
    )

    fig.show()

except Exception as e:
    print("Plotly nuk u ekzekutua në këtë mjedis.")
    print("Gabimi:", e)

# 5. Një shembull i shkurtër jashtë lëkundjeve  
## Hedhja balistike me rezistencë të ajrit

Metodat numerike për ODE nuk vlejnë vetëm për sisteme lëkundëse.  
Një shembull shumë i mirë, dhe intuitiv për studentët, është **hedhja balistike me rezistencë të ajrit**.

Marrim gjendjen:

\[
\mathbf y = (x, y, v_x, v_y),
\]

dhe modelin:

\[
\dot x = v_x,\qquad \dot y = v_y,
\]

\[
\dot v_x = -c\,v\,v_x,\qquad
\dot v_y = -g - c\,v\,v_y,
\]

ku \(v = \sqrt{v_x^2+v_y^2}\).

Ky shembull tregon qartë se **i njëjti aparat numerik** mund të përdoret në probleme shumë të ndryshme.

In [ ]:
def ode_ballistic_drag(t, y, g=9.81, c=0.03):
    x, yy, vx, vy = y
    v = np.sqrt(vx**2 + vy**2)
    ax = -c * v * vx
    ay = -g - c * v * vy
    return [vx, vy, ax, ay]

speed0 = 20.0
angle0 = np.deg2rad(45.0)

y0_ball = [0.0, 0.0, speed0*np.cos(angle0), speed0*np.sin(angle0)]
t_eval_ball = np.linspace(0, 4.0, 800)

sol_ball = solve_ivp(
    ode_ballistic_drag,
    (0.0, 4.0),
    y0_ball,
    t_eval=t_eval_ball,
    rtol=1e-8,
    atol=1e-10
)

mask = sol_ball.y[1] >= 0  # mbajmë vetëm pjesën mbi tokë

plt.figure(figsize=(8, 4))
plt.plot(sol_ball.y[0][mask], sol_ball.y[1][mask], label="Me rezistencë ajri")
plt.xlabel("x [m]")
plt.ylabel("y [m]")
plt.title("Hedhje balistike me rezistencë të ajrit")
plt.legend()
plt.show()

# 6. Ushtrime të propozuara për studentët

## Ushtrimi 1 — Mekanizmi bielë-manivelë me moment të jashtëm

Shtoni në model një moment periodik:

\[
	au(t)=	au_0\sin(\Omega t).
\]

Detyrat:

1. modifikoni ODE-në;
2. zgjidhni numerikisht për disa vlera të \(\Omega\);
3. krahasoni \(\theta(t)\) dhe \(x(t)\) për raste të ndryshme.

---

## Ushtrimi 2 — Penduli i dyfishtë me amortizim të lehtë

Shtoni në ekuacionet e pendulit të dyfishtë terma amortizues proporcionalë me \(\dot\theta_1\) dhe \(\dot\theta_2\).

Detyrat:

1. implementoni termat jo-konservativë;
2. studioni si ndryshon sjellja për amortizim të vogël, mesatar dhe të madh;
3. krahasoni trajektoren e masës së dytë në secilin rast.

---

## Ushtrimi 3 — Hedhja balistike pa dhe me rezistencë ajri

Zgjidhni dy modele:

- modeli ideal pa fërkim;
- modeli me rezistencë kuadratike.

Detyrat:

1. vizatoni të dy trajektoret në të njëjtin grafik;
2. gjeni lartësinë maksimale dhe largësinë horizontale;
3. diskutoni cili model është më realist.

---

## Ushtrimi 4 — Sistemi Lotka–Volterra

Shqyrtoni modelin:

\[
\dot x = lpha x - eta x y,
\qquad
\dot y = \delta x y - \gamma y.
\]

Detyrat:

1. zgjidhni numerikisht sistemin;
2. vizatoni \(x(t)\), \(y(t)\);
3. vizatoni portretin fazor \((x,y)\);
4. diskutoni pse ky është një shembull i sistemit dinamik jashtë mekanikës klasike.

---

## Ushtrimi 5 — Projekt i shkurtër

Zgjidhni një sistem tjetër me 1 ose 2 shkallë lirie dhe paraqiteni sipas skemës:

1. përshkrimi fizik;
2. zgjedhja e koordinatave;
3. ndërtimi i modelit;
4. derivimi simbolik me SymPy (kur është e mundur);
5. kalimi në ODE të rendit të parë;
6. zgjidhja numerike;
7. grafiqet dhe një animacion i shkurtër.

# 7. Përfundime

Nga ky notebook duhet të mbeten të qarta tre ide qendrore:

1. **Lagranzhiani** është një mjet sistematik për të kaluar nga përshkrimi fizik te ekuacionet e lëvizjes;
2. **SymPy** e bën shumë më të lehtë derivimin simbolik për sisteme jo-lineare;
3. **solve_ivp**, grafiqet dhe animacionet na lejojnë të kalojmë nga ekuacionet abstrakte te interpretimi fizik.

Ky është pikërisht tranzicioni që kërkon modelimi modern në fizikë:

\[
	ext{fizikë} \;\longrightarrow\; 	ext{matematikë} \;\longrightarrow\; 	ext{algoritëm} \;\longrightarrow\; 	ext{simulim} \;\longrightarrow\; 	ext{interpretim}.
\]